# 약관 PDF 재파싱 (페이지 단위 청킹)

2단 레이아웃 PDF에서 특약명 + 본문을 올바르게 묶어서 저장합니다.

## 전략
- 각 페이지에서 특약명 감지
- 특약명이 있는 페이지 → 새 청크 시작
- 특약명이 없는 페이지 → 이전 특약 청크에 이어 붙이기
- 최종 청크를 새 Chroma DB에 저장

## 출력 DB
- `insurance_chroma_db_v2` (자동차보험)
- `cancer_chroma_db_v2` (암보험)
- `teeth_chroma_db_v2` (치아보험)

In [10]:
# !pip install pdfplumber langchain-huggingface langchain-chroma

In [11]:
import re
import pdfplumber
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("⏳ BGE-m3 임베딩 모델 로드 중...")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
print("✅ 임베딩 모델 로드 완료")

⏳ BGE-m3 임베딩 모델 로드 중...
✅ 임베딩 모델 로드 완료


In [24]:
# ==========================================
# 특약명 감지 패턴
# ==========================================
PATTERN_AUTO = re.compile(
    r'\[\d+(?:-\d+)?\]\s+.+?특별약관|'
    r'제\d+편\s+.{2,30}|'
    r'제\d+장\s+.{2,30}'
)
 
PATTERN_CLAUSE = re.compile(
    r'^\d+-\d+\.\s+.+?특별약관|'
    r'^\d+\.\s+.+?특별약관'
)
 
# 섹션명으로 인식하면 안 되는 키워드
SKIP_KEYWORDS = ["* 비고", "비고", "준용 규정", "보상 내용", "적용 대상", "지급 기준"]
 
 
def detect_section_title(text: str, insurance_type: str) -> str | None:
    """
    페이지 텍스트에서 새 섹션(특약명) 감지
    감지되면 섹션명 반환, 없으면 None
    """
    lines = text.strip().split("\n")
    found_title = None
 
    for line in lines[:10]:  # 페이지 상단 10줄만 체크
        line = line.strip()
        if not line:
            continue
 
        # 섹션명으로 인식하면 안 되는 키워드 필터
        if any(line.startswith(kw) for kw in SKIP_KEYWORDS):
            continue
 
        if insurance_type == "auto":
            # * 비고, 비고 라인 skip
            if line.startswith("* 비고") or line.startswith("비고"):
                continue
            if PATTERN_AUTO.search(line):
                found_title = line
                break
            if re.match(r'^제\d+편|^제\d+장', line):
                found_title = line
                break
 
        else:  # cancer, teeth
            if PATTERN_CLAUSE.match(line):
                found_title = line
                break
            if re.match(r'^보통약관$', line):
                found_title = line
                break
 
    if found_title:
        # 1. 날짜 제거
        found_title = re.sub(r'\s+\d{4}년\s+\d+월\s+\d+일.*$', '', found_title).strip()
 
        # 2. "특별약관" 이후 본문 텍스트 제거
        match = re.search(r'(.+?특별약관)', found_title)
        if match:
            found_title = match.group(1).strip()
 
        return found_title
 
    return None
 
 
def parse_pdf_to_chunks(
    pdf_path: str,
    insurance_type: str,
    skip_pages: int = 5,
    max_chunk_chars: int = 6000,
) -> list:
    """
    PDF를 페이지 단위로 읽어서 특약명 기준으로 청킹
    """
    chunks = []
    current_title = "보통약관"
    current_text  = ""
    current_pages = []
 
    def flush_chunk():
        nonlocal current_text, current_pages
        text = current_text.strip()
        if len(text) >= 50:
            chunks.append(Document(
                page_content=f"[{current_title}]\n\n{text}",
                metadata={
                    "section": current_title,
                    "pages":   str(current_pages),
                    "type":    insurance_type,
                }
            ))
        current_text  = ""
        current_pages = []
 
    with pdfplumber.open(pdf_path) as pdf:
        total = len(pdf.pages)
        print(f"  총 {total}페이지 파싱 중...")
 
        for i, page in enumerate(pdf.pages):
            if i < skip_pages:
                continue
 
            text = page.extract_text() or ""
            if not text.strip():
                continue
 
            # 새 섹션 감지
            new_title = detect_section_title(text, insurance_type)
 
            if new_title:
                if current_text:
                    flush_chunk()
                current_title = new_title
 
            current_text += "\n" + text
            current_pages.append(i + 1)
 
            # 청크가 너무 길어지면 중간에 자르기
            if len(current_text) > max_chunk_chars:
                flush_chunk()
                current_title = current_title + " (계속)"
 
        flush_chunk()
 
    print(f"  총 {len(chunks)}개 청크 생성")
    return chunks
 
 
def build_db(
    pdf_path: str,
    insurance_type: str,
    dst_dir: str,
    db_name: str,
    skip_pages: int = 5,
    max_chunk_chars: int = 6000,
):
    """PDF 파싱 → 청킹 → Chroma DB 저장"""
    import os, shutil
 
    print(f"\n{'='*50}")
    print(f"[{db_name}] 파싱 시작")
    print(f"{'='*50}")
 
    chunks = parse_pdf_to_chunks(
        pdf_path=pdf_path,
        insurance_type=insurance_type,
        skip_pages=skip_pages,
        max_chunk_chars=max_chunk_chars,
    )
 
    # 기존 DB 삭제
    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir)
        print(f"  기존 {dst_dir} 삭제 완료")
 
    # 배치 임베딩
    batch_size = 50
    db = None
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]
        if db is None:
            db = Chroma.from_documents(
                documents=batch,
                embedding=embeddings,
                persist_directory=dst_dir
            )
        else:
            db.add_documents(batch)
        print(f"  임베딩 진행: {min(i+batch_size, len(chunks))}/{len(chunks)}개")
 
    print(f"✅ [{db_name}] 완료: {db._collection.count()}개 청크 저장")
    return db
 
print("✅ 함수 정의 완료")

✅ 함수 정의 완료


In [27]:
# ==========================================
# 자동차보험 파싱
# ==========================================
auto_db_v2 = build_db(
    pdf_path       = "./자동차보험_약관.pdf",
    insurance_type = "auto",
    dst_dir        = "./insurance_chroma_db_last",
    db_name        = "자동차보험",
    skip_pages     = 10,  # 표지/목차/안내문 skip
    max_chunk_chars = 6000
)


[자동차보험] 파싱 시작
  총 282페이지 파싱 중...
  총 74개 청크 생성
  임베딩 진행: 50/74개
  임베딩 진행: 74/74개
✅ [자동차보험] 완료: 74개 청크 저장


In [28]:
# ==========================================
# 암보험 파싱
# ==========================================
cancer_db_v2 = build_db(
    pdf_path       = "./암보험_약관.pdf",
    insurance_type = "cancer",
    dst_dir        = "./cancer_chroma_db_last",
    db_name        = "암보험",
    skip_pages     = 35,  # 목차/요약서/민원예시 skip
    max_chunk_chars = 6000
)


[암보험] 파싱 시작
  총 362페이지 파싱 중...
  총 115개 청크 생성
  임베딩 진행: 50/115개
  임베딩 진행: 100/115개
  임베딩 진행: 115/115개
✅ [암보험] 완료: 115개 청크 저장


In [29]:
# ==========================================
# 치아보험 파싱
# ==========================================
teeth_db_v2 = build_db(
    pdf_path       = "./치아보험_약관.pdf",
    insurance_type = "teeth",
    dst_dir        = "./teeth_chroma_db_last",
    db_name        = "치아보험",
    skip_pages     = 27,  # 목차/요약서/민원예시 skip
    max_chunk_chars = 6000
)


[치아보험] 파싱 시작
  총 201페이지 파싱 중...
  총 55개 청크 생성
  임베딩 진행: 50/55개
  임베딩 진행: 55/55개
✅ [치아보험] 완료: 55개 청크 저장


In [30]:
# ==========================================
# 검색 테스트
# ==========================================
tests = [
    (auto_db_v2,   "다른 자동차 운전담보 특별약관 보장 내용",   "자동차보험"),
    (auto_db_v2,   "음주운전 사고 면책금",                    "자동차보험"),
    (cancer_db_v2, "암 진단비 특별약관 보장 내용",             "암보험"),
    (cancer_db_v2, "유사암 진단비 지급 기준",                  "암보험"),
    (teeth_db_v2,  "치아보철 치료지원금 특별약관 보장 내용",    "치아보험"),
    (teeth_db_v2,  "임플란트 치조골 이식술 보장",               "치아보험"),
]

print("\n검색 테스트")
print("=" * 60)

for db, query, name in tests:
    print(f"\n[{name}] '{query}'")
    results = db.similarity_search(query, k=3)
    for i, r in enumerate(results):
        print(f"  [{i+1}] 섹션: {r.metadata.get('section', '-')}")
        print(f"       내용: {r.page_content[:250]}")
        print()


검색 테스트

[자동차보험] '다른 자동차 운전담보 특별약관 보장 내용'
  [1] 섹션: [71] 다른 자동차 운전담보 특별약관
       내용: [[71] 다른 자동차 운전담보 특별약관]

* 비고
[71] 다른 자동차 운전담보 특별약관
1. 적용 대상
이 특별약관은 보통약관 제2편 2장 2절 ‘무보험 자동차에 의한 상 다른 자동차란
해’에 가입한 경우에 한하여 가입할 수 있습니다. 자가용자동차로서 피보
험자동차와 동일한 차종
[승용자동차(일반 승용
2. 보상 내용 및 다목적 승용을포함합
니다), 경ㆍ3종승합자동
① 보험회사(이하 “회사”라 합니다)는 피보험자가 다른 자동차를 차 및 초

  [2] 섹션: [43-1] 다른 자동차 차량손해 자녀운전담보 추가 특별약관
       내용: [[43-1] 다른 자동차 차량손해 자녀운전담보 추가 특별약관]

* 비고 이 특별약관에서 정하지 아니한 사항은 보통약관에 따릅니다.
[43-1] 다른 자동차 차량손해 자녀운전담보 추가 특별약관
다른 자동차란
자가용자동차로서 피보
험자동차와 동일한 차종
[승용자동차(일반 승용 1. 가입 조건
및 다목적 승용을 포함합
이 추가특별약관은 ‘다른 자동차 차량손해 지원 특별약관’에 가입한
니다), 경·3종승합자동차
및 초소형·경·4종화물자 경우에 한하

  [3] 섹션: [71-1] 다른 자동차 자녀운전담보 추가 특별약관
       내용: [[71-1] 다른 자동차 자녀운전담보 추가 특별약관]

* 비고 이 특별약관에서 피보험자란 기명 피보험자 및 기명 피보험자의 배
우자를 말합니다. 다만, 기명 피보험자의 배우자는 운전자를 한정
하는 다른 특별약관에 의하여 운전 가능 범위에 포함되지 않는 경우
에는 피보험자로 보지 아니합니다.
5. 준용 규정
이 특별약관에서 정하지 아니한 사항은 보통약관에 따릅니다.
[71-1] 다른 자동차 자녀운전담보 추가 특별약관
다른 자동차란
자가용자동차


[자동차보험] '음주운전 사고 면책금'
  [1] 섹션: 제1장 보험계약의 성립 (계속) (계속

## 검색 결과 확인 후

결과가 잘 나오면 `llm_setup.py`에서 DB 경로를 `_v2`로 변경하세요.

```python
auto_db      = Chroma(persist_directory="./insurance_chroma_db_v2", embedding_function=embeddings)
cancer_db    = Chroma(persist_directory="./cancer_chroma_db_v2",    embedding_function=embeddings)
teeth_db     = Chroma(persist_directory="./teeth_chroma_db_v2",     embedding_function=embeddings)
```